## [节点配置](https://docs.langchain.com/oss/javascript/langgraph/graph-api-nodes)

### [超时处理](https://docs.langchain.com/oss/python/langgraph/fault-tolerance#timeouts)

#### 超时策略

- run_timeout: 节点执行超时时间
- idle_timeout: 节点空闲超时时间

In [ ]:
from typing import TypedDict

from langgraph.errors import NodeTimeoutError
from langgraph.graph import StateGraph, START, END
from langgraph.types import TimeoutPolicy


class State(TypedDict):
    pass

async def long_task(state: State):
    print("a")
    await asyncio.sleep(10)
    return {}


builder = StateGraph(State)
builder.add_node(
    long_task,
    timeout=TimeoutPolicy(run_timeout=5),
)
builder.add_edge(START, "long_task")
builder.add_edge("long_task", END)
graph = builder.compile()

try:
    await graph.ainvoke({})
except NodeTimeoutError:
    print("节点执行超时")

refresh_on: 决定 idle_timeout 在什么情况下重新计时

In [6]:
import asyncio
from typing import TypedDict

from langgraph.graph import END, START, StateGraph
from langgraph.runtime import Runtime
from langgraph.types import TimeoutPolicy


class State(TypedDict):
    result: str

async def long_task(state: State, runtime: Runtime):
    for i in range(5):
        await asyncio.sleep(1)
        print(f"已完成 {i + 1}/5")

        # 报告进度，刷新 idle_timeout
        runtime.heartbeat()
    return {"result": "完成"}


builder = StateGraph(State)
builder.add_node(
    long_task,
    timeout=TimeoutPolicy(
        idle_timeout=2,
        run_timeout=8,
        refresh_on="heartbeat",
    ),
)
builder.add_edge(START, "long_task")
builder.add_edge("long_task", END)

graph = builder.compile()
result = await graph.ainvoke({})
print(result)

已完成 1/5
已完成 2/5
已完成 3/5
已完成 4/5
已完成 5/5
{'result': '完成'}
